# Model Traning

In [ ]:
import pandas as pd
import numpy as np

#  Load the data
data = pd.read_csv('../data/synthetic_retail_cleaned.csv')


In [ ]:
data.head()

,TransactionID,Date,Time,LaborHours,InventoryCost,FootTraffic,CustomerID,SalesAmount,LaborProductivity,InventoryEfficiency,FootTrafficEfficiency
0,1,1/1/2026,12:00:00 AM,7,296,65,1,580.021831,82.860262,1.959533,8.923413
1,2,1/1/2026,1:00:00 AM,4,155,49,2,378.975444,94.743861,2.445003,7.734193
2,3,1/1/2026,2:00:00 AM,5,100,107,3,747.893378,149.578676,7.478934,6.989658
3,4,1/1/2026,3:00:00 AM,7,130,49,4,412.560604,58.937229,3.173543,8.419604
4,5,1/1/2026,4:00:00 AM,3,182,100,5,704.820557,234.940186,3.872640,7.048206


##### Feature Engineering

In [ ]:
# Date se Month aur Day nikalna
data['Date'] = pd.to_datetime(data['Date'])
data['Month'] = data['Date'].dt.month
data['DayOfWeek'] = data['Date'].dt.dayofweek

# Time se Hour nikalna
data['Hour'] = pd.to_datetime(data['Time']).dt.hour


C:\Users\Muhammad Umair\AppData\Local\Temp\ipykernel_11860\129466437.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Hour'] = pd.to_datetime(data['Time']).dt.hour


In [ ]:
# Direct columns ke naam likh kar drop karein
data.drop(columns=['CustomerID','TransactionID','Date','Time', 'LaborProductivity', 'InventoryEfficiency', 'FootTrafficEfficiency'], inplace=True, errors='ignore')

# Check karne ke liye
print(data.columns)


Index(['LaborHours', 'InventoryCost', 'FootTraffic', 'SalesAmount', 'Month',
       'DayOfWeek', 'Hour'],
      dtype='object')


In [ ]:
# Save karein
data.to_csv('../Data/Model_Data.csv', index=False)

In [ ]:
data.head()

,LaborHours,InventoryCost,FootTraffic,SalesAmount,Month,DayOfWeek,Hour
0,7,296,65,580.021831,1,3,0
1,4,155,49,378.975444,1,3,1
2,5,100,107,747.893378,1,3,2
3,7,130,49,412.560604,1,3,3
4,3,182,100,704.820557,1,3,4


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# In columns ko select karein
X = data[['LaborHours', 'InventoryCost', 'FootTraffic', 'Month', 'DayOfWeek', 'Hour']]
y = data['SalesAmount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = RandomForestRegressor()
model.fit(X_train, y_train)

predictions = model.predict(X_test)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print("Model Performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

Model Performance:
MAE: 31.095189347457485
RMSE: 38.91162435954206
R2 Score: 0.9875394547071341


In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/SalesPrediction.pkl')

print("Model saved successfully!")

Model saved successfully!


In [ ]:
data.head()

,LaborHours,InventoryCost,FootTraffic,SalesAmount,Month,DayOfWeek,Hour
0,7,296,65,580.021831,1,3,0
1,4,155,49,378.975444,1,3,1
2,5,100,107,747.893378,1,3,2
3,7,130,49,412.560604,1,3,3
4,3,182,100,704.820557,1,3,4


In [ ]:
# ... (after your model.fit line) ...

# 1. Predict for the ENTIRE dataset (not just the test set)
data['Predicted_SalesAmount'] = model.predict(data[['LaborHours', 'InventoryCost', 'FootTraffic', 'Month', 'DayOfWeek', 'Hour']])

# 2. (Optional) Calculate the Error/Difference for Power BI analysis
data['Prediction_Error'] = data['SalesAmount'] - data['Predicted_SalesAmount']

# 3. Check the first few rows
print(data[['SalesAmount', 'Predicted_SalesAmount', 'Prediction_Error']].head())

# 4. Save to CSV so Power BI can read the new columns
data.to_csv('../Data/Sales_Report_with_Predictions.csv', index=False)


   SalesAmount  Predicted_SalesAmount  Prediction_Error
0   580.021831             583.773064         -3.751233
1   378.975444             385.839015         -6.863571
2   747.893378             759.401538        -11.508160
3   412.560604             417.845544         -5.284941
4   704.820557             697.933504          6.887052
